In [ ]:
import ee
import geemap
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, GridSearchCV, StratifiedKFold
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
from sklearn.preprocessing import StandardScaler
from sklearn.feature_selection import SelectKBest, f_classif
import xgboost as xgb
import joblib
import seaborn as sns
import matplotlib.pyplot as plt
from typing import List, Tuple
import warnings
from datetime import datetime
warnings.filterwarnings('ignore')

In [ ]:
# Name of project in google cloud available to use with Earth Engine
EE_PROJECT = "ee-gis-scode"

try:
    ee.Authenticate()
    if EE_PROJECT:
        ee.Initialize(project=EE_PROJECT)
    else:
        ee.Initialize()
except Exception as e:
    raise RuntimeError(
        "Failed to initialize the Earth Engine client. "
    ) from e

In [ ]:
# --- 1. Definir Área de Interés (AOI) ---
# Using FAO GAUL dataset to get country boundaries
countries_dataset = ee.FeatureCollection("FAO/GAUL_SIMPLIFIED_500m/2015/level2");

cuenca_ecuador = countries_dataset.filter(ee.Filter.eq('ADM1_NAME', 'Azuay')) \
                                .filter(ee.Filter.eq('ADM2_NAME', 'Cuenca'))

cuenca_aoi = cuenca_ecuador.geometry()

Map = geemap.Map()
Map.centerObject(cuenca_aoi, 10)
Map.addLayer(cuenca_aoi, {'color':'red'}, "Cuenca AOI")
display(Map)

In [ ]:
# --- CONSTANTES GLOBALES (Se asume que cuenca_aoi ya está definida) ---
CLOUD_SCORE_COLLECTION = 'GOOGLE/CLOUD_SCORE_PLUS/V1/S2_HARMONIZED'
SENTINEL_COLLECTION = 'COPERNICUS/S2_SR_HARMONIZED'
MAX_CLOUD_PERCENTAGE = 35
CLEAR_THRESHOLD = 0.6  # Umbral de claridad (0.6 = 60% claro)

# --- 1. FUNCIÓN DE ENMASCARAMIENTO (Cloud Score Plus) ---
def mask_s2_cs_plus(image):
    """
    Aplica una máscara de nubes y sombras usando la banda 'cs' y reescala las bandas.
    """
    # 1. Crear la máscara: cs >= 0.6
    qaBand = 'cs'
    mask = image.select(qaBand).gte(CLEAR_THRESHOLD)
    
    # 2. Reescalar y aplicar la máscara (S2_SR está en 0-10000, dividimos por 10000)
    return image.updateMask(mask).divide(10000)

# --- 2. FUNCIÓN PARA CREAR CUALQUIER COMPOSICIÓN ANUAL LIBRE DE NUBES ---
def get_annual_composite(year: int, aoi: ee.Geometry) -> ee.Image:
    """
    Crea una composición mediana (medoid) libre de nubes para un año específico
    usando el método linkCollection y el enmascaramiento Cloud Score Plus.
    """
    start_date = f'{year}-01-01'
    end_date = f'{year}-12-31'
    
    # a. Colección base (S2_SR)
    s2_collection = ee.ImageCollection(SENTINEL_COLLECTION)\
        .filterDate(start_date, end_date)\
        .filterBounds(aoi)\
        .filter(ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE', MAX_CLOUD_PERCENTAGE))

    # b. Colección de scores (CSP) y unión (linkCollection)
    cs_plus = ee.ImageCollection(CLOUD_SCORE_COLLECTION).filterDate(start_date, end_date)
    collection_with_cs = s2_collection.linkCollection(cs_plus, ['cs'])

    # c. Aplicar Máscara y Composición Mediana
    collection_sin_nubes = collection_with_cs.map(mask_s2_cs_plus)
    median_composite = collection_sin_nubes.median().clip(aoi)
    
    return median_composite
# =========================================
# 2. NUEVA INTEGRACIÓN: TENDENCIA HISTÓRICA (CORREGIDO)
# =========================================
# Función auxiliar para preparar pares [Año, NDVI] para linearFit
def create_year_ndvi(year):
    # Nota: Usamos get_annual_composite que ya tienes definida
    comp = get_annual_composite(year, cuenca_aoi)
    ndvi = comp.normalizedDifference(['B8', 'B4']).rename('ndvi')
    # Creamos una imagen constante con el valor del año (Variable X)
    year_img = ee.Image.constant(year).toFloat().rename('year')
    # Retornamos la imagen con bandas: [year, ndvi] (ORDEN IMPORTANTE PARA LINEARFIT)
    return year_img.addBands(ndvi)

# Creamos una colección con los 5 años
years_list = [2020, 2021, 2022, 2023, 2024]
trend_collection = ee.ImageCollection.fromImages([create_year_ndvi(y) for y in years_list])

# Calculamos la tendencia.
# linearFit usa la banda 0 como X (year) y la banda 1 como Y (ndvi)
ndvi_trend = trend_collection.reduce(ee.Reducer.linearFit()).select('scale').rename('NDVI_trend')
print("Tendencia histórica (2020-2024) calculada.")
# --- 3. CREACIÓN DE TODAS LAS CARACTERÍSTICAS (SPECTRALES, TEMPORALES, TOPOGRÁFICAS) ---
print("--- INICIANDO CREACIÓN DE CARACTERÍSTICAS MEJORADAS ---")

# 3.1. Composiciones Mediana para múltiples años (Mejora: más años para mejor análisis temporal)
print("--- Creando composiciones anuales (2020-2024) ---")
composite_2024 = get_annual_composite(2024, cuenca_aoi)
composite_2023 = get_annual_composite(2023, cuenca_aoi)
composite_2022 = get_annual_composite(2022, cuenca_aoi)
composite_2021 = get_annual_composite(2021, cuenca_aoi)
composite_2020 = get_annual_composite(2020, cuenca_aoi)
print("Composiciones 2020-2024 listas.")

# 3.2. CARACTERÍSTICAS TEMPORALES MEJORADAS
# Múltiples comparaciones temporales para capturar mejor los cambios
ndvi_2024 = composite_2024.normalizedDifference(['B8', 'B4']).rename('NDVI_2024')
ndvi_2023 = composite_2023.normalizedDifference(['B8', 'B4']).rename('NDVI_2023')
ndvi_2022 = composite_2022.normalizedDifference(['B8', 'B4']).rename('NDVI_2022')
ndvi_2021 = composite_2021.normalizedDifference(['B8', 'B4']).rename('NDVI_2021')
ndvi_2020 = composite_2020.normalizedDifference(['B8', 'B4']).rename('NDVI_2020')

# Diferencia entre años consecutivos (múltiples comparaciones)
ndvi_diff_2024_2023 = ndvi_2024.subtract(ndvi_2023).rename('NDVI_diff_2024_2023')
ndvi_diff_2023_2022 = ndvi_2023.subtract(ndvi_2022).rename('NDVI_diff_2023_2022')
ndvi_diff_2024_2022 = ndvi_2024.subtract(ndvi_2022).rename('NDVI_diff_2024_2022')
ndvi_diff_2024_2020 = ndvi_2024.subtract(ndvi_2020).rename('NDVI_diff_2024_2020')

# Ratio de cambio (más robusto que diferencia)
ndvi_ratio_2024_2023 = ndvi_2024.divide(ndvi_2023.add(0.001)).rename('NDVI_ratio_2024_2023')

# Media móvil de NDVI (suaviza el ruido)
ndvi_mean_3yr = ndvi_2024.add(ndvi_2023).add(ndvi_2022).divide(3).rename('NDVI_mean_3yr')

print("Características temporales mejoradas calculadas.")


# 3.3. CARACTERÍSTICAS DE ÍNDICES ESPECTRALES MEJORADAS (Basado en 2024)
# Más índices para mejor discriminación
b2 = composite_2024.select('B2').rename('Blue')
b3 = composite_2024.select('B3').rename('Green')
b4 = composite_2024.select('B4').rename('Red')
b8 = composite_2024.select('B8').rename('NIR')
b11 = composite_2024.select('B11').rename('SWIR1')  # SWIR para mejor detección
b12 = composite_2024.select('B12').rename('SWIR2')

# Índices básicos
ndvi = b8.subtract(b4).divide(b8.add(b4)).rename('NDVI')
evi = composite_2024.expression(
    '2.5 * ((NIR - RED) / (NIR + 6 * RED - 7.5 * BLUE + 1))', {
        'NIR': b8, 'RED': b4, 'BLUE': b2
    }).rename('EVI')
savi = composite_2024.expression(
    '((NIR - RED) / (NIR + RED + 0.5)) * 1.5', {
        'NIR': b8, 'RED': b4
    }).rename('SAVI')
ndwi = b3.subtract(b8).divide(b3.add(b8)).rename('NDWI')

# NUEVOS ÍNDICES PARA MEJOR PRECISIÓN
# NBR (Normalized Burn Ratio) - excelente para detectar cambios en vegetación
nbr = composite_2024.expression(
    '(NIR - SWIR1) / (NIR + SWIR1)', {
        'NIR': b8, 'SWIR1': b11
    }).rename('NBR')

# MNDWI (Modified NDWI) - mejor detección de agua
mndwi = composite_2024.expression(
    '(GREEN - SWIR1) / (GREEN + SWIR1)', {
        'GREEN': b3, 'SWIR1': b11
    }).rename('MNDWI')

# GNDVI (Green NDVI) - más sensible a clorofila
gndvi = composite_2024.expression(
    '(NIR - GREEN) / (NIR + GREEN)', {
        'NIR': b8, 'GREEN': b3
    }).rename('GNDVI')

# NDMI (Normalized Difference Moisture Index) - contenido de humedad
ndmi = composite_2024.expression(
    '(NIR - SWIR1) / (NIR + SWIR1)', {
        'NIR': b8, 'SWIR1': b11
    }).rename('NDMI')

# BSI (Bare Soil Index) - detecta suelo desnudo
bsi = composite_2024.expression(
    '((RED + SWIR1) - (NIR + BLUE)) / ((RED + SWIR1) + (NIR + BLUE))', {
        'RED': b4, 'SWIR1': b11, 'NIR': b8, 'BLUE': b2
    }).rename('BSI')

# EVI2 (simplificado, menos dependiente del azul)
evi2 = composite_2024.expression(
    '2.5 * ((NIR - RED) / (NIR + 2.4 * RED + 1))', {
        'NIR': b8, 'RED': b4
    }).rename('EVI2')

print("Índices espectrales mejorados calculados (NDVI, EVI, SAVI, NDWI, NBR, MNDWI, GNDVI, NDMI, BSI, EVI2).")


# 3.4. CARACTERÍSTICAS AUXILIARES (TOPOGRAFÍA Y TEXTURA)

# TOPOGRAFÍA (DEM y Slope)
# Aunque parezcan irrelevantes, la topografía ayuda a contextualizar si el área
# clasificada como "Deforestado" está en un sitio donde la deforestación es común.
dem = ee.Image('USGS/SRTMGL1_003').clip(cuenca_aoi)
elevation = dem.select('elevation')
slope = ee.Terrain.slope(dem).rename('slope')
print("Características de topografía listas.")# Extraemos las dos "capas" que nos importan de la super-imagen.
#treecover2000: Una capa que muestra el porcentaje de cobertura de árboles en el año 2000. La usaremos para definir dónde había bosque originalmente.
#lossyear: Una capa donde el valor de cada píxel indica el año en que se perdió el bosque (si es que se perdió). Un valor de 23 significa que la pérdida ocurrió en 2023.

# TEXTURA (Entropy de GLCM)
# La textura (entropía) diferencia un bosque denso (uniforme) de un área
# deforestada y fragmentada (alta entropía).
# Se usa NIR (B8) y se convierte a entero para la función GLCM.
def calculate_glcm_features(band_image: ee.Image, band_name: str, window_size=4) -> ee.Image:
    """
    Calcula múltiples características de textura GLCM (Entropía, Contraste, Homogeneidad, Varianza)
    para una banda de entrada específica, renombrando las bandas de salida con un prefijo.
    """
    # Multiplicar y convertir a entero (int16) para que GLCM funcione correctamente.
    band_int = band_image.multiply(1000).toInt16()
    
    # Calcular las texturas.
    glcm = band_int.glcmTexture(size=window_size)
    
    # Extraer y renombrar las características de textura.
    entropy = glcm.select(f'{band_name}_ent').rename(f'{band_name}_entropy')
    contrast = glcm.select(f'{band_name}_contrast').rename(f'{band_name}_contrast')
    homogeneity = glcm.select(f'{band_name}_idm').rename(f'{band_name}_homogeneity')
    variance = glcm.select(f'{band_name}_var').rename(f'{band_name}_variance')
    
    # Concatenar las cuatro características en una sola imagen de bandas múltiples.
    return ee.Image.cat([entropy, contrast, homogeneity, variance])

# nir_integer = b8.multiply(1000).toInt16()
# glcm = nir_integer.glcmTexture(size=4)
# entropy = glcm.select('NIR_ent').rename('entropy') # Usamos el nuevo nombre 'NIR'
glcm_nir = calculate_glcm_features(b8, 'NIR')
glcm_red = calculate_glcm_features(b4, 'Red')
glcm_ndvi = calculate_glcm_features(ndvi, 'NDVI')

print("Característica de textura (Entropy) calculada.")

# 3.5. COMBINAR Y SELECCIONAR BANDAS FINALES

# Lista de todas las bandas a incluir en el modelo (MEJORADA con más características)
BANDS_TO_SELECT = [
    # Bandas espectrales básicas
    'Blue', 'Green', 'Red', 'NIR', 'SWIR1', 'SWIR2',
    # Índices de vegetación (básicos)
    'NDVI', 'EVI', 'SAVI', 'NDWI',
    # Índices mejorados
    'NBR', 'MNDWI', 'GNDVI', 'NDMI', 'BSI', 'EVI2',
    # Características temporales mejoradas
    'NDVI_2024', 'NDVI_2023', 'NDVI_2022',
    'NDVI_diff_2024_2023', 'NDVI_diff_2023_2022', 'NDVI_diff_2024_2022', 'NDVI_diff_2024_2020',
    'NDVI_ratio_2024_2023', 'NDVI_mean_3yr', 'NDVI_trend',
    # Topografía
    'elevation', 'slope',
    # Textura (GLCM) - múltiples bandas
    'NIR_entropy', 'NIR_contrast', 'NIR_homogeneity', 'NIR_variance',
    'Red_entropy', 'Red_contrast', 'Red_homogeneity', 'Red_variance',
    'NDVI_entropy', 'NDVI_contrast', 'NDVI_homogeneity', 'NDVI_variance'
]

# Juntamos todas las características en una sola imagen.
final_features = composite_2024.select(['B2', 'B3', 'B4', 'B8', 'B11', 'B12'])\
    .rename(['Blue', 'Green', 'Red', 'NIR', 'SWIR1', 'SWIR2'])\
    .addBands([
        # Índices básicos
        ndvi, evi, savi, ndwi,
        # Índices mejorados
        nbr, mndwi, gndvi, ndmi, bsi, evi2,
        # Características temporales
        ndvi_2024, ndvi_2023, ndvi_2022,
        ndvi_diff_2024_2023, ndvi_diff_2023_2022, ndvi_diff_2024_2022, ndvi_diff_2024_2020,
        ndvi_ratio_2024_2023, ndvi_mean_3yr, ndvi_trend,
        # Topografía
        elevation, slope,
        # Textura
        glcm_nir, glcm_red, glcm_ndvi
    ])

# Aseguramos que solo tengamos las bandas deseadas y en el orden correcto
final_features = final_features.select(BANDS_TO_SELECT)


print("\n--- ¡PROCESO FINALIZADO! ---")
print("Imagen final de características creada con todas las bandas.")
print("Bandas disponibles para el modelo:", final_features.bandNames().getInfo())

In [ ]:
# --- (Todo tu código de la Celda 7 va aquí arriba) ---
# ...
# final_features = final_features.select(BANDS_TO_SELECT)
# print("Imagen final de características creada con todas las bandas.")
# ...

# --- 8. EXPORTAR IMAGEN DE CARACTERÍSTICAS A GOOGLE DRIVE ---
# Este es el paso de "Inferencia" para aplicar el modelo localmente.

print("\n--- INICIANDO EXPORTACIÓN A GOOGLE DRIVE ---")

# Define la geometría de exportación
export_region = cuenca_aoi

# Define el nombre del archivo y la carpeta en tu Google Drive
file_name = 'cuenca_features_2024_para_prediccion'
export_folder = 'GEE_Exports' # (Asegúrate de que esta carpeta exista en tu Drive)

# Configuración de la tarea de exportación
task = ee.batch.Export.image.toDrive(
    image=final_features.toFloat(), # Convertir a float32 para GeoTiff
    description=file_name,
    folder=export_folder,
    fileNamePrefix=file_name,
    region=export_region,
    scale=10,             # ¡MUY IMPORTANTE! Usar la escala de 10m de Sentinel-2
    crs='EPSG:32717',     # CRS Proyectado para Cuenca (UTM 17S)
    maxPixels=1e13        # Permitir exportaciones grandes
)

# Iniciar la tarea
task.start()

print(f"✅ Tarea de exportación '{file_name}' iniciada.")
print(f"   Revisa la pestaña 'Tasks' en Google Earth Engine para monitorear el progreso.")
print(f"   El archivo se guardará en tu Google Drive en la carpeta: '{export_folder}'")

# --- ¡ADVERTENCIA DE PACIENCIA! ---
print("\n" + "="*50)
print("¡PACIENCIA! ESTA TAREA TARDARÁ MUCHO TIEMPO")
print("Has solicitado 40 bandas complejas (incluyendo texturas y 5 años de datos)")
print("a 10m de resolución. GEE puede tardar varias horas en procesar y")
print("exportar este archivo GeoTiff.")
print("="*50)